In [9]:
from quant.data.repository import QuantRepository
from quant.data.db import DuckDBManager
from quant.config import load_config
import datetime

config = load_config()
raw_dir = config.paths.raw_dir
process_dir = config.paths.processed_dir
database_path = config.paths.database_path
dbManager = DuckDBManager(database_path, process_dir)
dbManager.initialize()
with dbManager.session() as conn:
    qr = QuantRepository(conn)
    td = qr.get_trade_calendar(datetime.date(2026,1,1),datetime.date(2026,6,20))
    print(td)


2026-06-20 21:02:55.724 | INFO     | quant.data.db:_register_parquet_views:141 - 已注册 DuckDB 视图 v_daily_ohlcv


2026-06-20 21:02:55.724 | DEBUG    | quant.data.db:_register_parquet_views:128 - 跳过缺失的 Parquet 数据集 adj_factor
2026-06-20 21:02:55.724 | DEBUG    | quant.data.db:_register_parquet_views:128 - 跳过缺失的 Parquet 数据集 daily_basic
2026-06-20 21:02:55.725 | DEBUG    | quant.data.db:_register_parquet_views:128 - 跳过缺失的 Parquet 数据集 index_daily
2026-06-20 21:02:55.727 | DEBUG    | quant.data.db:_register_parquet_views:128 - 跳过缺失的 Parquet 数据集 fundamental
2026-06-20 21:02:55.727 | DEBUG    | quant.data.db:_register_parquet_views:128 - 跳过缺失的 Parquet 数据集 factors


[{'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 1), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 2), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 3), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 4), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 5), 'is_open': True, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 6), 'is_open': True, 'pretrade_date': datetime.date(2026, 1, 5)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 7), 'is_open': True, 'pretrade_date': datetime.date(2026, 1, 6)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 8), 'is_open': True, 'pretrade_date': datetime.date(2026, 1, 7)}, {'exchange': 'SSE

## 检查 daily-ohlcv raw 中的 BJ 证券交易日

直接扫描 `data/raw/tushare/daily-ohlcv/**/*.csv`, 查询每年出现 `.BJ` 证券的交易日。第一段 SQL 做年度汇总, 第二段 SQL 展开到具体交易日。

In [10]:
import duckdb

from quant.config import load_config

config = load_config()
raw_glob = (
    config.paths.raw_dir
    / "tushare"
    / "daily-ohlcv"
    / "**"
    / "*.csv"
).as_posix().replace("'", "''")

# 年度汇总: 每年有多少个交易日出现过 .BJ 证券。
bj_yearly_sql = f"""
WITH bj_daily AS (
    SELECT
        ts_code,
        strptime(trade_date, '%Y%m%d')::DATE AS trade_date
    FROM read_csv_auto(
        '{raw_glob}',
        all_varchar = true,
        union_by_name = true,
        filename = true
    )
    WHERE ts_code LIKE '%.BJ'
)
SELECT
    EXTRACT(YEAR FROM trade_date)::INTEGER AS trade_year,
    MIN(trade_date) AS first_trade_date,
    MAX(trade_date) AS last_trade_date,
    COUNT(DISTINCT trade_date) AS trade_date_count,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ts_code) AS security_count
FROM bj_daily
GROUP BY trade_year
ORDER BY trade_year;
"""

duckdb.sql(bj_yearly_sql)


┌────────────┬──────────────────┬─────────────────┬──────────────────┬───────────┬────────────────┐
│ trade_year │ first_trade_date │ last_trade_date │ trade_date_count │ row_count │ security_count │
│   int32    │       date       │      date       │      int64       │   int64   │     int64      │
├────────────┼──────────────────┼─────────────────┼──────────────────┼───────────┼────────────────┤
│       2013 │ 2013-01-18       │ 2013-12-17      │               39 │        40 │              4 │
│       2014 │ 2014-01-03       │ 2014-12-31      │              151 │       358 │             17 │
│       2015 │ 2015-01-05       │ 2015-12-31      │              244 │      4874 │             73 │
│       2016 │ 2016-01-04       │ 2016-12-30      │              244 │     10655 │            124 │
│       2017 │ 2017-01-03       │ 2017-12-29      │              244 │     10431 │            152 │
│       2018 │ 2018-01-02       │ 2018-12-28      │              243 │      6952 │            156 │


In [11]:
# 明细: 展开到每一个含 .BJ 证券的交易日。
bj_trade_dates_sql = f"""
WITH bj_daily AS (
    SELECT
        ts_code,
        strptime(trade_date, '%Y%m%d')::DATE AS trade_date
    FROM read_csv_auto(
        '{raw_glob}',
        all_varchar = true,
        union_by_name = true,
        filename = true
    )
    WHERE ts_code LIKE '%.BJ'
)
SELECT
    EXTRACT(YEAR FROM trade_date)::INTEGER AS trade_year,
    trade_date,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ts_code) AS security_count,
    string_agg(DISTINCT ts_code, ', ' ORDER BY ts_code) AS bj_codes
FROM bj_daily
GROUP BY trade_year, trade_date
ORDER BY trade_date;
"""

duckdb.sql(bj_trade_dates_sql)


┌────────────┬────────────┬───────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────